In [1]:
import pandas as pd
import json
from collections import Counter
from pathlib import Path


df_keybert = pd.read_csv("../dataset_comparative_2020_2026/comparative_files/keywords_keybert_meta.csv")
print(f"Документів: {len(df_keybert):,}")
df_keybert.head()

Документів: 8,427


,doc_id,length,rubric,outcome,num_stages,main_committee,card_id,reg_num,reg_date,keywords_keybert,num_keywords
0,12273_Порівняльна таблиця (04.12.2024),172,Економічна політика,rejected,5,Підласа Р. А. Комітет Верховної Ради України з...,45350,12273,2024-12-02,"[""бюджет україни"", ""україни 2023"", ""закон укра...",5.0
1,12272_Порівняльна таблиця (04.12.2024),376,Економічна політика,in_progress,4,Підласа Р. А. Комітет Верховної Ради України з...,45328,12272,2024-12-02,"[""бюджетний кодекс"", ""україни відомості"", ""зді...",5.0
2,12274_Порівняльна таблиця (03.12.2024),145,Економічна політика,rejected,5,Підласа Р. А. Комітет Верховної Ради України з...,45349,12274,2024-12-02,"[""бюджет україни"", ""україни 2023"", ""закон укра...",5.0
3,12275_Порівняльна таблиця (04.12.2024),476,Безпека і оборона,in_progress,4,Завітневич О. М. Комітет Верховної Ради Україн...,45341,12275,2024-12-02,"[""закону україни"", ""закон україни"", ""законами ...",5.0
4,12276_Порівняльна таблиця (04.12.2024),499,Економічна політика,in_progress,4,Гетманцев Д. О. Комітет Верховної Ради України...,45363,12276,2024-12-02,"[""особливостей оподаткування"", ""податку стаття...",5.0


In [2]:
from collections import defaultdict
import json

# Підрахунок document frequency для KeyBERT
doc_freq = defaultdict(int)
for kw_json in df_keybert["keywords_keybert"]:
    try:
        kw_list = json.loads(kw_json)
        unique_kw = set(
            kw.lower().strip()
            for kw in kw_list
        )
        for kw in unique_kw:
            doc_freq[kw] += 1
    except:
        pass

total_docs = len(df_keybert)
print(f"Всього документів: {total_docs}\n")
print(f"{'KW':<40} {'DF':>6} {'DF%':>7}")
print("-" * 60)
for kw, df_count in sorted(
    doc_freq.items(),
    key=lambda x: -x[1]
)[:50]:
    pct = df_count / total_docs * 100
    print(
        f"{kw:<40} "
        f"{df_count:>6} "
        f"{pct:>6.1f}%"
    )

Всього документів: 8427

KW                                           DF     DF%
------------------------------------------------------------
закону україни                             3801   45.1%
закон україни                              2508   29.8%
законів україни                            1775   21.1%
кодексу україни                            1526   18.1%
порівняльна таблиця                        1121   13.3%
міністрів україни                           981   11.6%
законом україни                             929   11.0%
бюджету україни                             707    8.4%
кодекс україни                              681    8.1%
україни підписувач                          628    7.5%
депутати україни                            616    7.3%
бюджет україни                              569    6.8%
україни стаття                              525    6.2%
норми проекту                               514    6.1%
актів україни                               504    6.0%
проекту закону    

In [3]:
all_kw_keybert = []
for kw_json in df_keybert["keywords_keybert"]:
    try:
        all_kw_keybert.extend([kw.lower().strip() for kw in json.loads(kw_json)])
    except:
        pass

counter_keybert = Counter(all_kw_keybert)
print(f"Унікальних KW: {len(counter_keybert):,}")
print(f"Всього KW: {len(all_kw_keybert):,}")
print(f"\nТоп-30:")
for kw, cnt in counter_keybert.most_common(30):
    print(f"  {cnt:>5}  {kw}")

Унікальних KW: 66,552
Всього KW: 176,349

Топ-30:
   3801  закону україни
   2508  закон україни
   1775  законів україни
   1526  кодексу україни
   1121  порівняльна таблиця
    981  міністрів україни
    929  законом україни
    707  бюджету україни
    681  кодекс україни
    628  україни підписувач
    616  депутати україни
    569  бюджет україни
    525  україни стаття
    514  норми проекту
    504  актів україни
    484  проекту закону
    479  законодавства зміст
    471  ради україни
    451  депутат україни
    440  конституції україни
    390  республіки крим
    347  законами україни
    338  україни закон
    315  україни реєстраційний
    302  безпеки україни
    301  законодавства україни
    292  рада україни
    290  адміністративні правопорушення
    283  деяких законодавчих
    277  конституцією україни


In [4]:
### ФІЛЬТРАЦІЯ

In [5]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import json

BASE_DIR = Path("../dataset_comparative_2020_2026")

input_path = BASE_DIR / "comparative_files" / "keywords_keybert_meta.csv"

OUTPUT_DIR = BASE_DIR / "comparative_files" / "keywords_keybert_filtered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEYWORD_COL = "keywords_keybert"

df = pd.read_csv(input_path)

df_original = df.copy()
total_docs = len(df_original)

print(f"Документів: {total_docs:,}")
print(df_original.columns.tolist())

Документів: 8,427
['doc_id', 'length', 'rubric', 'outcome', 'num_stages', 'main_committee', 'card_id', 'reg_num', 'reg_date', 'keywords_keybert', 'num_keywords']


In [6]:
def parse_keywords_keybert(kw_json):
    try:
        kw_list = json.loads(kw_json)

        keywords = []

        for item in kw_list:
            if isinstance(item, list):
                kw = item[0]
            else:
                kw = item

            kw = str(kw).lower().strip()

            if kw:
                keywords.append(kw)

        return keywords

    except:
        return []

In [7]:
def get_top_keywords_by_documents(df_input, keywords_col=KEYWORD_COL, top_n=30):
    doc_freq = defaultdict(int)
    total_docs = len(df_input)

    for kw_json in df_input[keywords_col]:
        keywords = parse_keywords_keybert(kw_json)

        unique_kw = set(keywords)

        for kw in unique_kw:
            doc_freq[kw] += 1

    rows = []

    for kw, cnt in sorted(doc_freq.items(), key=lambda x: -x[1])[:top_n]:
        rows.append({
            "keyword": kw,
            "documents": cnt,
            "percent_of_docs": round(cnt / total_docs * 100, 2)
        })

    return pd.DataFrame(rows)
    
top_before_filter = get_top_keywords_by_documents(
    df_original,
    keywords_col=KEYWORD_COL,
    top_n=30
)

print("ТОП ключових слів ДО фільтрації > 9%:")
display(top_before_filter)

ТОП ключових слів ДО фільтрації > 9%:


,keyword,documents,percent_of_docs
0,закону україни,3801,45.11
1,закон україни,2508,29.76
2,законів україни,1775,21.06
3,кодексу україни,1526,18.11
4,порівняльна таблиця,1121,13.30
5,міністрів україни,981,11.64
6,законом україни,929,11.02
7,бюджету україни,707,8.39
8,кодекс україни,681,8.08
9,україни підписувач,628,7.45


In [8]:
doc_freq = defaultdict(int)

for kw_json in df_original[KEYWORD_COL]:
    keywords = parse_keywords_keybert(kw_json)

    unique_kw = set(keywords)

    for kw in unique_kw:
        doc_freq[kw] += 1

print(f"Унікальних ключових слів: {len(doc_freq):,}")

Унікальних ключових слів: 66,552


In [9]:
def filter_keywords_keybert_9pct(kw_json):
    keywords = parse_keywords_keybert(kw_json)

    filtered = [
        kw
        for kw in keywords
        if kw not in stopwords_9pct
    ]

    return json.dumps(filtered, ensure_ascii=False)

In [13]:
from collections import defaultdict
import pandas as pd
import json

doc_freq = defaultdict(int)
total_docs = len(df_original)

for kw_json in df_original[KEYWORD_COL]:
    keywords = parse_keywords_keybert(kw_json)

    unique_kw = set(keywords)

    for kw in unique_kw:
        doc_freq[kw] += 1

DF_THRESHOLD = 0.09

stopwords_9pct = {
    kw
    for kw, cnt in doc_freq.items()
    if cnt / total_docs > DF_THRESHOLD
}

print(f"Всього документів: {total_docs:,}")
print(f"Унікальних ключових слів: {len(doc_freq):,}")
print(f"Слів, які зустрічаються більше ніж у 9% документів: {len(stopwords_9pct):,}")

Всього документів: 8,427
Унікальних ключових слів: 66,552
Слів, які зустрічаються більше ніж у 9% документів: 7


In [14]:
df_keywords_filtered = df_original.copy()

df_keywords_filtered["keywords_keybert_original"] = df_keywords_filtered[KEYWORD_COL]

df_keywords_filtered[KEYWORD_COL] = df_keywords_filtered[KEYWORD_COL].apply(
    filter_keywords_keybert_9pct
)

df_keywords_filtered["num_keywords_after_9pct_filter"] = df_keywords_filtered[KEYWORD_COL].apply(
    lambda x: len(parse_keywords_keybert(x))
)

print("Середня кількість KW після фільтрації:")
print(round(df_keywords_filtered["num_keywords_after_9pct_filter"].mean(), 2))

display(df_keywords_filtered.head())

Середня кількість KW після фільтрації:
19.43


,doc_id,length,rubric,outcome,num_stages,main_committee,card_id,reg_num,reg_date,keywords_keybert,num_keywords,keywords_keybert_original,num_keywords_after_9pct_filter
0,12273_Порівняльна таблиця (04.12.2024),172,Економічна політика,rejected,5,Підласа Р. А. Комітет Верховної Ради України з...,45350,12273,2024-12-02,"[""бюджет україни"", ""україни 2023"", ""бюджету 20...",5.0,"[""бюджет україни"", ""україни 2023"", ""закон укра...",3
1,12272_Порівняльна таблиця (04.12.2024),376,Економічна політика,in_progress,4,Підласа Р. А. Комітет Верховної Ради України з...,45328,12272,2024-12-02,"[""бюджетний кодекс"", ""україни відомості"", ""зді...",5.0,"[""бюджетний кодекс"", ""україни відомості"", ""зді...",5
2,12274_Порівняльна таблиця (03.12.2024),145,Економічна політика,rejected,5,Підласа Р. А. Комітет Верховної Ради України з...,45349,12274,2024-12-02,"[""бюджет україни"", ""україни 2023""]",5.0,"[""бюджет україни"", ""україни 2023"", ""закон укра...",2
3,12275_Порівняльна таблиця (04.12.2024),476,Безпека і оборона,in_progress,4,Завітневич О. М. Комітет Верховної Ради Україн...,45341,12275,2024-12-02,"[""законами україни"", ""прав військовослужбовців...",5.0,"[""закону україни"", ""закон україни"", ""законами ...",3
4,12276_Порівняльна таблиця (04.12.2024),499,Економічна політика,in_progress,4,Гетманцев Д. О. Комітет Верховної Ради України...,45363,12276,2024-12-02,"[""особливостей оподаткування"", ""податку стаття...",5.0,"[""особливостей оподаткування"", ""податку стаття...",5


In [15]:
top_after_filter = get_top_keywords_by_documents(
    df_keywords_filtered,
    keywords_col=KEYWORD_COL,
    top_n=30
)

print("ТОП ключових слів ПІСЛЯ фільтрації >9%:")
display(top_after_filter)

ТОП ключових слів ПІСЛЯ фільтрації >9%:


,keyword,documents,percent_of_docs
0,бюджету україни,707,8.39
1,кодекс україни,681,8.08
2,україни підписувач,628,7.45
3,депутати україни,616,7.31
4,бюджет україни,569,6.75
5,україни стаття,525,6.23
6,норми проекту,514,6.10
7,актів україни,504,5.98
8,проекту закону,484,5.74
9,законодавства зміст,479,5.68


In [16]:
top_before_filter_compare = top_before_filter.rename(columns={
    "keyword": "keyword_before",
    "documents": "documents_before",
    "percent_of_docs": "percent_before"
})

top_after_filter_compare = top_after_filter.rename(columns={
    "keyword": "keyword_after",
    "documents": "documents_after",
    "percent_of_docs": "percent_after"
})

top_keywords_before_after = pd.concat(
    [top_before_filter_compare, top_after_filter_compare],
    axis=1
)

display(top_keywords_before_after)

,keyword_before,documents_before,percent_before,keyword_after,documents_after,percent_after
0,закону україни,3801,45.11,бюджету україни,707,8.39
1,закон україни,2508,29.76,кодекс україни,681,8.08
2,законів україни,1775,21.06,україни підписувач,628,7.45
3,кодексу україни,1526,18.11,депутати україни,616,7.31
4,порівняльна таблиця,1121,13.30,бюджет україни,569,6.75
5,міністрів україни,981,11.64,україни стаття,525,6.23
6,законом україни,929,11.02,норми проекту,514,6.10
7,бюджету україни,707,8.39,актів україни,504,5.98
8,кодекс україни,681,8.08,проекту закону,484,5.74
9,україни підписувач,628,7.45,законодавства зміст,479,5.68


In [17]:
filtered_csv = OUTPUT_DIR / "keywords_keybert_filtered.csv"
filtered_parquet = OUTPUT_DIR / "keywords_keybert_filtered.parquet"

df_keywords_filtered.to_csv(filtered_csv, index=False, encoding="utf-8-sig")
df_keywords_filtered.to_parquet(filtered_parquet, index=False)

print("Збережено:")
print(filtered_csv)
print(filtered_parquet)

Збережено:
..\dataset_comparative_2020_2026\comparative_files\keywords_keybert_filtered\keywords_keybert_filtered.csv
..\dataset_comparative_2020_2026\comparative_files\keywords_keybert_filtered\keywords_keybert_filtered.parquet


In [18]:
FILTER_STEPS = [0.02, 0.04, 0.06, 0.08, 0.10]

filtered_versions = {}

for p in FILTER_STEPS:

    threshold = df_keywords_filtered["length"].quantile(p)

    df_length_filtered = df_keywords_filtered[
        df_keywords_filtered["length"] > threshold
    ].copy()

    filter_name = f"without_shortest_{int(p * 100)}pct"
    filtered_versions[filter_name] = df_length_filtered

    csv_path = OUTPUT_DIR / f"keywords_keybert_filtered_{int(p * 100)}pct.csv"
    parquet_path = OUTPUT_DIR / f"keywords_keybert_filtered_{int(p * 100)}pct.parquet"

    df_length_filtered.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df_length_filtered.to_parquet(parquet_path, index=False)

    print("-" * 60)
    print(f"Відкинуто найменші {int(p * 100)}% документів")
    print(f"Поріг length: {threshold:.2f}")
    print(f"Залишилось документів: {len(df_length_filtered):,}")
    print(f"Saved CSV     : {csv_path}")
    print(f"Saved Parquet : {parquet_path}")

------------------------------------------------------------
Відкинуто найменші 2% документів
Поріг length: 139.00
Залишилось документів: 8,257
Saved CSV     : ..\dataset_comparative_2020_2026\comparative_files\keywords_keybert_filtered\keywords_keybert_filtered_2pct.csv
Saved Parquet : ..\dataset_comparative_2020_2026\comparative_files\keywords_keybert_filtered\keywords_keybert_filtered_2pct.parquet
------------------------------------------------------------
Відкинуто найменші 4% документів
Поріг length: 170.00
Залишилось документів: 8,082
Saved CSV     : ..\dataset_comparative_2020_2026\comparative_files\keywords_keybert_filtered\keywords_keybert_filtered_4pct.csv
Saved Parquet : ..\dataset_comparative_2020_2026\comparative_files\keywords_keybert_filtered\keywords_keybert_filtered_4pct.parquet
------------------------------------------------------------
Відкинуто найменші 6% документів
Поріг length: 189.00
Залишилось документів: 7,918
Saved CSV     : ..\dataset_comparative_2020_2026